# EuroSAT CNN — Transfer Learning Pipeline

Land-use classification on Sentinel-2 satellite imagery using 5 pretrained CNN architectures.

**Pipeline:**  
1. Generate stratified train/val/test splits  
2. Exploratory data analysis  
3. Train all 5 models (VGG16, ResNet50, DenseNet121, EfficientNet-B0, MobileNetV2)  
4. Evaluate & compare

**Hardware:** Colab GPU (T4 / V100 / A100)

## 0. Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/AhmetServet/eurosat-cnn.git
%cd eurosat-cnn

In [ ]:
# Upload dataset.zip to Colab first, then unzip
# (Or mount Google Drive and copy from there)
from google.colab import files, drive
import zipfile, os

# Option A: Upload zip manually
# uploaded = files.upload()
# for filename in uploaded:
#     with zipfile.ZipFile(filename, 'r') as z:
#         z.extractall('dataset/')

# Option B: Mount Google Drive (if dataset is there)
drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/eurosat_dataset/* dataset/

print('Dataset ready:', len(os.listdir('dataset')))

In [ ]:
# Install dependencies
!pip install -q torch torchvision scikit-learn pandas matplotlib seaborn tqdm pyyaml pillow numpy

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 1. Generate Train / Val / Test Splits

In [ ]:
!python scripts/01_generate_csv.py

In [ ]:
# Verify splits
import pandas as pd
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'splits/{split}.csv')
    print(f"{split}: {len(df)} samples")
print("\nClass distribution (train):")
print(pd.read_csv('splits/train.csv')['className'].value_counts().sort_index())

## 2. Exploratory Data Analysis

In [ ]:
!python scripts/02_eda.py

In [ ]:
# Display EDA outputs
from IPython.display import Image, display
print("Class Distribution:")
display(Image('outputs/eda/class_distribution.png'))

print("\nSample Grid:")
display(Image('outputs/eda/sample_grid.png'))

---
## 3. Train All Models

Trains 5 architectures with frozen ImageNet backbones.  
Each model: ~5-15 min on T4 GPU. Full run: ~1-2 hours.

**To resume after Colab disconnects:** just re-run this cell — checkpoints auto-resume.

In [ ]:
# Train all 5 models
# Uses: AMP (mixed precision), torch.compile, early stopping, checkpoint/resume
!python scripts/03_train.py --all

In [ ]:
# Check training progress — show training curves for completed models
from IPython.display import Image as IPImage, display
from pathlib import Path
import json

ckpt_dir = Path('outputs/checkpoints')
for p in sorted(ckpt_dir.glob('*_history.json')):
    name = p.stem.replace('_history', '')
    hist = json.loads(p.read_text())
    print(f"{name}: best epoch={hist.get('best_epoch','?')}, "
          f"val_loss={hist.get('best_val_loss','?')}")

---
## 4. Evaluate & Compare

In [ ]:
!python scripts/04_evaluate.py

In [ ]:
# Display evaluation results
from IPython.display import Image as IPImage, display
from pathlib import Path
import pandas as pd
import json

# Comparison table
df = pd.read_csv('outputs/report/model_comparison.csv')
print("\n=== Model Comparison ===")
display(df)

# Comparison chart
display(IPImage('outputs/report/model_comparison.png'))

# Confusion matrices for each model
for p in sorted(Path('outputs/plots').glob('*_confusion.png')):
    name = p.stem.replace('_confusion', '')
    print(f"\n=== {name} — Confusion Matrix ===")
    display(IPImage(str(p)))

# ROC curves
for p in sorted(Path('outputs/plots').glob('*_roc.png')):
    name = p.stem.replace('_roc', '')
    print(f"\n=== {name} — ROC Curves ===")
    display(IPImage(str(p)))

In [ ]:
# Save checkpoints & outputs to Google Drive
!cp -r outputs/ /content/drive/MyDrive/eurosat_outputs/
print("Saved to Google Drive ✓")

---

## Done

All models trained and evaluated. Check `outputs/report/model_comparison.csv` for the final ranking.

**To restart/resume:** re-run Section 0 (mount drive, restore checkpoints), then jump to Section 3.